# Library

In [79]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# Load Clean Dataset

In [81]:
df = pd.read_parquet(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset/final_df_(distance).parquet"
)

final_features = [
    "total_price",
    "total_freight_value",
    "total_product_weight_g",
    "average_product_length_cm",
    "average_product_height_cm",
    "item_count",
    "max_payment_installments",
    "average_distance_km"
]

experiment_columns = [
    "order_id",
    *final_features,
    "is_delayed"
]

model_df = df[experiment_columns].dropna().copy()

print(model_df.shape)
print(model_df.isna().sum().sum())

(95960, 10)
0


In [82]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=42,
    stratify=model_df["is_delayed"]
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (76768, 10)
Test: (19192, 10)


In [83]:
train_idx = train_df.index
test_idx = test_df.index

print("Train indices:", len(train_idx))
print("Test indices:", len(test_idx))

Train indices: 76768
Test indices: 19192


# Load Missing Dataset

In [84]:
from pathlib import Path

data_folder = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/exp_data")

injection_data = {}

for file in data_folder.glob("*.parquet"):
    injection_data[file.stem] = pd.read_parquet(file)

print(injection_data.keys())

dict_keys(['mar_50', 'mnar_50', 'mcar_30', 'mar_30', 'mnar_30', 'mcar_50', 'mcar_10', 'mar_10', 'mnar_10'])


In [29]:
mcar_10 = injection_data["mcar_10"]

In [31]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.2,
    random_state=42,
    stratify=model_df["is_delayed"]
)

# Model function

In [ ]:
def run_xgb_and_record(
        train_df,
        test_df,
        mechanism,
        missing_rate,
        method
):
    X_train = train_df[final_features]
    y_train = train_df['is_delayed']

    X_test = test_df[final_features]
    y_test = test_df['is_delayed']

    xgb_model = XGBClassifier(
        n_estimators = 100,
        max_depth =5,
        learning_rate =0.1,
        random_state =42,
        eval_metric ='logloss',
        n_jobs =-1
    )

    xgb_model.fit(X_train,y_train)

    

# MCAR

## Mean_10

In [33]:
mcar_10.index = model_df.index

In [34]:
mcar_10_train = mcar_10.loc[train_idx].copy()
mcar_10_test = mcar_10.loc[test_idx].copy()

print("Train:", mcar_10_train.shape)
print("Test:", mcar_10_test.shape)

Train: (76768, 10)
Test: (19192, 10)


In [35]:
mean_value = mcar_10_train["total_freight_value"].mean()

mcar_10_train["total_freight_value"] = (
    mcar_10_train["total_freight_value"].fillna(mean_value)
)

mcar_10_test["total_freight_value"] = (
    mcar_10_test["total_freight_value"].fillna(mean_value)
)

print("Mean value:", mean_value)
print("Train missing:", mcar_10_train["total_freight_value"].isna().sum())
print("Test missing:", mcar_10_test["total_freight_value"].isna().sum())

Mean value: 22.821277830018087
Train missing: 0
Test missing: 0


In [36]:
features = final_features

X_train = mcar_10_train[features]
y_train = mcar_10_train["is_delayed"]

X_test = mcar_10_test[features]
y_test = mcar_10_test["is_delayed"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Missing in X_train:", X_train.isna().sum().sum())
print("Missing in X_test:", X_test.isna().sum().sum())

X_train: (76768, 8)
X_test: (19192, 8)
Missing in X_train: 0
Missing in X_test: 0


In [57]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

xgb_model = XGBClassifier(
    n_estimators = 100,
    max_depth = 5,
    learning_rate = 0.1,
    random_state =42,
    eval_metric = 'logloss',
    n_jobs = -1
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]


In [58]:
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_xgb, zero_division=0))
print("F1:", f1_score(y_test, y_pred_xgb, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

Accuracy: 0.9189766569403919
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.6031013797907679


In [59]:
downstream_results = []

In [60]:
downstream_results.append({
    "mechanism": "MCAR",
    "missing_rate": 0.10,
    "method": "Mean",
    "accuracy": accuracy_score(y_test, y_pred_xgb),
    "precision": precision_score(y_test, y_pred_xgb, zero_division=0),
    "recall": recall_score(y_test, y_pred_xgb, zero_division=0),
    "f1": f1_score(y_test, y_pred_xgb, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob_xgb)
})

In [61]:
downstream_results

[{'mechanism': 'MCAR',
  'missing_rate': 0.1,
  'method': 'Mean',
  'accuracy': 0.9189766569403919,
  'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0,
  'roc_auc': 0.6031013797907679}]

## Median_10

In [62]:
mcar_10_train = mcar_10.loc[train_idx].copy()
mcar_10_test = mcar_10.loc[test_idx].copy()

In [63]:
median_value = mcar_10_train["total_freight_value"].median()

mcar_10_train["total_freight_value"] = (
    mcar_10_train["total_freight_value"].fillna(median_value)
)

mcar_10_test["total_freight_value"] = (
    mcar_10_test["total_freight_value"].fillna(median_value)
)

print("Median value:", median_value)
print("Train missing:", mcar_10_train["total_freight_value"].isna().sum())
print("Test missing:", mcar_10_test["total_freight_value"].isna().sum())

Median value: 17.22
Train missing: 0
Test missing: 0


In [ ]:
X_train = mcar_10_train[final_features]
y_train = mcar_10_train["is_delayed"]

X_test = mcar_10_test[final_features]
y_test = mcar_10_test["is_delayed"]

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9149124635264694
Precision: 0.15178571428571427
Recall: 0.010932475884244373
F1: 0.020395920815836834
ROC-AUC: 0.5841719222614984


## KNN_10

In [65]:
mcar_10_train = mcar_10.loc[train_idx].copy()
mcar_10_test = mcar_10.loc[test_idx].copy()

In [66]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
imputer = KNNImputer(n_neighbors=5, weights="uniform")

X_train_missing = mcar_10_train[final_features].copy()
X_test_missing = mcar_10_test[final_features].copy()

X_train_scaled = scaler.fit_transform(X_train_missing)
X_test_scaled = scaler.transform(X_test_missing)

X_train_imputed = imputer.fit_transform(X_train_scaled)
X_test_imputed = imputer.transform(X_test_scaled)

X_train_imputed = pd.DataFrame(
    X_train_imputed,
    columns=final_features,
    index=X_train_missing.index
)

X_test_imputed = pd.DataFrame(
    X_test_imputed,
    columns=final_features,
    index=X_test_missing.index
)

print("Train missing:", X_train_imputed.isna().sum().sum())
print("Test missing:", X_test_imputed.isna().sum().sum())

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: in

Train missing: 0
Test missing: 0


In [67]:
y_train = mcar_10_train["is_delayed"]
y_test = mcar_10_test["is_delayed"]

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train_imputed, y_train)

y_pred = rf.predict(X_test_imputed)
y_prob = rf.predict_proba(X_test_imputed)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9152250937890788
Precision: 0.17272727272727273
Recall: 0.012218649517684888
F1: 0.022822822822822823
ROC-AUC: 0.5819722568766663


In [68]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mcar_10_train = mcar_10.loc[train_idx].copy()
mcar_10_test = mcar_10.loc[test_idx].copy()

X_train_missing = mcar_10_train[final_features].copy()
X_test_missing = mcar_10_test[final_features].copy()

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_missing)
X_test_scaled = scaler.transform(X_test_missing)

mice = IterativeImputer(
    max_iter=10,
    random_state=42
)

X_train_imputed_scaled = mice.fit_transform(X_train_scaled)
X_test_imputed_scaled = mice.transform(X_test_scaled)

X_train_imputed = pd.DataFrame(
    scaler.inverse_transform(X_train_imputed_scaled),
    columns=final_features,
    index=X_train_missing.index
)

X_test_imputed = pd.DataFrame(
    scaler.inverse_transform(X_test_imputed_scaled),
    columns=final_features,
    index=X_test_missing.index
)

print("Train missing:", X_train_imputed.isna().sum().sum())
print("Test missing:", X_test_imputed.isna().sum().sum())

Train missing: 0
Test missing: 0


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

In [69]:
import numpy as np

print("Train finite:", np.isfinite(X_train_imputed.to_numpy()).all())
print("Test finite:", np.isfinite(X_test_imputed.to_numpy()).all())

print("\nTrain range:")
print(X_train_imputed[final_features].agg(["min", "max"]).T)

print("\nTest range:")
print(X_test_imputed[final_features].agg(["min", "max"]).T)

Train finite: True
Test finite: True

Train range:
                            min            max
total_price                0.85   13440.000000
total_freight_value        0.00    1794.960000
total_product_weight_g     0.00  154200.000000
average_product_length_cm  7.00     105.000000
average_product_height_cm  2.00     105.000000
item_count                 1.00      21.000000
max_payment_installments   0.00      26.000000
average_distance_km        0.00    8677.911622

Test range:
                            min            max
total_price                0.85    5934.600000
total_freight_value        0.00     711.330000
total_product_weight_g     0.00  184400.000000
average_product_length_cm  7.00     105.000000
average_product_height_cm  2.00     105.000000
item_count                 1.00      15.000000
max_payment_installments   1.00      24.000000
average_distance_km        0.00    3381.363478


In [70]:
non_target_features = [
    feature
    for feature in final_features
    if feature != "total_freight_value"
]

for feature in non_target_features:
    train_diff = (
        X_train_imputed[feature]
        - X_train_missing[feature]
    ).abs().max()

    test_diff = (
        X_test_imputed[feature]
        - X_test_missing[feature]
    ).abs().max()

    print(
        feature,
        "Train max diff:", train_diff,
        "Test max diff:", test_diff
    )

total_price Train max diff: 4.547473508864641e-13 Test max diff: 4.547473508864641e-13
total_product_weight_g Train max diff: 1.4551915228366852e-11 Test max diff: 7.275957614183426e-12
average_product_length_cm Train max diff: 0.0 Test max diff: 0.0
average_product_height_cm Train max diff: 1.4210854715202004e-14 Test max diff: 1.4210854715202004e-14
item_count Train max diff: 3.552713678800501e-15 Test max diff: 0.0
max_payment_installments Train max diff: 3.552713678800501e-15 Test max diff: 1.7763568394002505e-15
average_distance_km Train max diff: 4.547473508864641e-13 Test max diff: 4.547473508864641e-13


In [71]:
y_train = mcar_10_train["is_delayed"]
y_test = mcar_10_test["is_delayed"]

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train_imputed, y_train)

y_pred = rf.predict(X_test_imputed)
y_prob = rf.predict_proba(X_test_imputed)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.915068778657774
Precision: 0.1559633027522936
Recall: 0.010932475884244373
F1: 0.020432692307692308
ROC-AUC: 0.585421250670224


### Results

In [72]:
downstream_results = []

In [73]:
def record_downstream_result(
    results,
    mechanism,
    missing_rate,
    method,
    y_test,
    y_pred,
    y_prob
):
    results.append({
        "mechanism": mechanism,
        "missing_rate": missing_rate,
        "method": method,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob)
    })

In [74]:
record_downstream_result(
    downstream_results,
    mechanism="MCAR",
    missing_rate=0.10,
    method="Mean",
    y_test=y_test,
    y_pred=y_pred,   # 注意：这里必须是 Mean 那次 RF 的 prediction
    y_prob=y_prob
)

In [75]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train_imputed, y_train)

y_pred = rf.predict(X_test_imputed)
y_prob = rf.predict_proba(X_test_imputed)[:, 1]

record_downstream_result(
    downstream_results,
    mechanism="MCAR",
    missing_rate=0.10,
    method="MICE",
    y_test=y_test,
    y_pred=y_pred,
    y_prob=y_prob
)

In [76]:
downstream_results_df = pd.DataFrame(downstream_results)

downstream_results_df

,mechanism,missing_rate,method,accuracy,precision,recall,f1,roc_auc
0,MCAR,0.1,Mean,0.915069,0.155963,0.010932,0.020433,0.585421
1,MCAR,0.1,MICE,0.915069,0.155963,0.010932,0.020433,0.585421


In [77]:
final_features
train_idx
test_idx

Index([45568, 37173, 68369, 34460, 61619, 75753, 23793, 56390, 98959, 93135,
       ...
       86592, 73264, 83960, 81798, 71837, 73841, 17383, 51135, 46813,  8053],
      dtype='int64', length=19192)

In [78]:
print(final_features)
print(len(train_idx))
print(len(test_idx))

['total_price', 'total_freight_value', 'total_product_weight_g', 'average_product_length_cm', 'average_product_height_cm', 'item_count', 'max_payment_installments', 'average_distance_km']
76768
19192
